## Import Libraries

In [ ]:
import numpy as np
import cv2

from pathlib import Path
import matplotlib.pyplot as plt
%matplotlib inline

import tensorflow as tf
from tensorflow.keras.layers import Conv2D, MaxPool2D, BatchNormalization, \
Activation, Flatten
from tensorflow.keras import models
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.optimizers import Adam

## Fetching Data & Hyper-parameter configuration

In [ ]:
# Number of epochs
EPOCHS = 50

# batch size
BATCH_SIZE = 8

# Random seed
SEED = 145

MODEL = "Customized_Model"

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
!unzip '/content/drive/My Drive/data/tomato_leaf_images.zip'

Archive:  /content/drive/My Drive/data/tomato_leaf_images.zip
   creating: tomato_leaf_images/train/
   creating: tomato_leaf_images/train/AmericanLeafMiner/
  inflating: tomato_leaf_images/train/AmericanLeafMiner/2217_256.jpg  
  inflating: tomato_leaf_images/train/AmericanLeafMiner/2227_256.jpg  
  inflating: tomato_leaf_images/train/AmericanLeafMiner/2247_256.jpg  
  inflating: tomato_leaf_images/train/AmericanLeafMiner/2252_256.jpg  
  inflating: tomato_leaf_images/train/AmericanLeafMiner/2254_256.jpg  
  inflating: tomato_leaf_images/train/AmericanLeafMiner/2255_256.jpg  
  inflating: tomato_leaf_images/train/AmericanLeafMiner/2260_256.jpg  
  inflating: tomato_leaf_images/train/AmericanLeafMiner/2261_256.jpg  
  inflating: tomato_leaf_images/train/AmericanLeafMiner/2262_256.jpg  
  inflating: tomato_leaf_images/train/AmericanLeafMiner/IMG_20190203_112440_256.jpg  
  inflating: tomato_leaf_images/train/AmericanLeafMiner/IMG_20190203_112500_256.jpg  
  inflating: tomato_leaf_images

The folder structure would be as shown here

In [ ]:
Tomato_leaf_images
...train
......AmericanLeafMiner
.........image_1.jpg
.........image_2.jpg
......Healthy
.........image_3.jpg
.........image_4.jpg
......MagnesiumDeficiency
.........image_5.jpg
.........image_6.jpg
......SerpentineLeafMiner
.........image_7.jpg
.........image_8.jpg

...val
......AmericanLeafMiner
.........image_9.jpg
......Healthy
.........image_10.jpg
......MagnesiumDeficiency
.........image_11.jpg
......SerpentineLeafMiner
.........image_12.jpg


In [ ]:
# Path of the data
image_train_dir =  'tomato_leaf_images/train'
image_test_dir = 'tomato_leaf_images/val'

num_classes = 4
target_size = (256,256)

## Helper functions

In [ ]:
# Function to get all images as one array

def image_to_array(data_dir, target_size):

    all_image_path = [str(path) for path in Path(data_dir).glob('*/*')]
    img_array = np.empty(shape=(len(all_image_path),256,256,3))

    for i, image in enumerate(all_image_path):
        img = cv2.cvtColor(cv2.imread(image), cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, target_size)
        img_array[i,:,:,:] = img

    return img_array

## Calculate data mean and sigma

In [ ]:
train_img_array = image_to_array(image_train_dir, target_size)
test_img_array = image_to_array(image_test_dir, target_size)

print('Shape of Train images array:',train_img_array.shape)
print('Shape of Test images array:',test_img_array.shape)

Channelwise_mean_train = np.mean(train_img_array, axis = (0,1,2))
Channelwise_std_train  = np.std(train_img_array, axis = (0,1,2))

print('Channelwise Mean of train images:', Channelwise_mean_train)
print('Channelwise std of train images:', Channelwise_std_train)

Shape of Train images array: (4356, 256, 256, 3)
Shape of Test images array: (486, 256, 256, 3)
Channelwise Mean of train images: [120.92365635 133.39954958  86.61648463]
Channelwise std of train images: [51.43527712 45.59955441 53.82409601]


## Data Generators (including data augmentation)

In [ ]:
# Image Augmentation - CutOut

def apply_cutout(img):
    n_holes = 1
    length = target_size[0] / 4

    img = np.array(img)
    h = img.shape[0]
    w = img.shape[1]

    mask = np.ones((h, w, 3), np.float32)

    for n in range(n_holes):
        y = np.random.randint(h)
        x = np.random.randint(w)

        y1 = int(np.clip(y - length // 2, 0, h))
        y2 = int(np.clip(y + length // 2, 0, h))
        x1 = int(np.clip(x - length // 2, 0, w))
        x2 = int(np.clip(x + length // 2, 0, w))

        mask[y1: y2, x1: x2] = 0. # implementation of cutout

    img = img * mask

    return img

In [ ]:
augmented_train_datagen = ImageDataGenerator(featurewise_center=True,
                                             rotation_range=90,
                                             brightness_range=[0.5,1.6],
                                             zoom_range=[0.5,1.0],
                                             horizontal_flip=True,
                                             vertical_flip=True,
                                             preprocessing_function=apply_cutout
                                            )
augmented_train_datagen.mean = Channelwise_mean_train
augmented_train_datagen.std = Channelwise_std_train

train_generator = augmented_train_datagen.flow_from_directory(image_train_dir,
                                                              target_size=(256,256),
                                                              batch_size=BATCH_SIZE,
                                                              class_mode='categorical')

# test generator
test_datagen = ImageDataGenerator(featurewise_center=True)

test_datagen.mean = Channelwise_mean_train
test_datagen.std = Channelwise_std_train

test_generator = test_datagen.flow_from_directory(image_test_dir,
                                                  target_size=(256,256),
                                                  batch_size=BATCH_SIZE,
                                                  class_mode='categorical')

Found 4356 images belonging to 4 classes.
Found 486 images belonging to 4 classes.


## Model Definition

In [ ]:
def get_model(custom_activation):
    """
    Function to get keras network with custom arguments
    :param custom_activation: str, or keras activation function
    return: keras model
    """

    model = models.Sequential()

    # Convolution
    model.add(Conv2D(filters=16, kernel_size=(3, 3), input_shape=(256, 256, 3)))
    model.add(BatchNormalization())
    model.add(Activation(custom_activation))

    model.add(Conv2D(filters=32, kernel_size=(3, 3)))
    model.add(BatchNormalization())
    model.add(Activation(custom_activation))

    model.add(Conv2D(filters=64, kernel_size=(3, 3)))
    model.add(BatchNormalization())
    model.add(Activation(custom_activation))

    model.add(Conv2D(filters=128, kernel_size=(3, 3)))
    model.add(BatchNormalization())
    model.add(Activation(custom_activation))

    model.add(Conv2D(filters=256, kernel_size=(3, 3)))
    model.add(BatchNormalization())
    model.add(Activation(custom_activation))

    # 1x1 convolution
    model.add(Conv2D(filters=64, kernel_size=(1, 1)))
    model.add(BatchNormalization())
    model.add(Activation(custom_activation))

    # Maxpooling
    model.add(MaxPool2D(pool_size=(2, 2)))

    # Convolution
    model.add(Conv2D(filters=64, kernel_size=(3, 3)))
    model.add(BatchNormalization())
    model.add(Activation(custom_activation))

    model.add(Conv2D(filters=128, kernel_size=(3, 3)))
    model.add(BatchNormalization())
    model.add(Activation(custom_activation))

    model.add(Conv2D(filters=256, kernel_size=(3, 3)))
    model.add(BatchNormalization())
    model.add(Activation(custom_activation))

    # 1x1 convolution
    model.add(Conv2D(filters=64, kernel_size=(1, 1)))
    model.add(BatchNormalization())
    model.add(Activation(custom_activation))
    # Maxpooling
    model.add(MaxPool2D(pool_size=(2, 2)))

    # Convolution
    model.add(Conv2D(filters=64, kernel_size=(3, 3)))
    model.add(BatchNormalization())
    model.add(Activation(custom_activation))

    model.add(Conv2D(filters=128, kernel_size=(3, 3)))
    model.add(BatchNormalization())
    model.add(Activation(custom_activation))

    model.add(Conv2D(filters=256, kernel_size=(3, 3)))
    model.add(BatchNormalization())
    model.add(Activation(custom_activation))

    # 1x1 convolution
    model.add(Conv2D(filters=64, kernel_size=(1, 1)))
    model.add(BatchNormalization())
    model.add(Activation(custom_activation))
    # Maxpooling
    model.add(MaxPool2D(pool_size=(2, 2)))

    # Convolution
    model.add(Conv2D(filters=64, kernel_size=(3, 3)))
    model.add(BatchNormalization())
    model.add(Activation(custom_activation))

    model.add(Conv2D(filters=32, kernel_size=(3, 3)))
    model.add(BatchNormalization())
    model.add(Activation(custom_activation))

    # 1x1 convolution
    model.add(Conv2D(filters=16, kernel_size=(1, 1)))
    model.add(BatchNormalization())
    model.add(Activation(custom_activation))
    # Maxpooling
    model.add(MaxPool2D(pool_size=(2, 2)))

    # Convolution
    model.add(Conv2D(filters=num_classes, kernel_size=(11, 11)))
    model.add(BatchNormalization())

    model.add(Flatten())
    model.add(Activation('softmax'))

    return model

#------------------#
model_v1 = get_model('relu')
model_v1.summary()

Model: "sequential_1"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
conv2d_18 (Conv2D)           (None, 254, 254, 16)      448       
_________________________________________________________________
batch_normalization_18 (Batc (None, 254, 254, 16)      64        
_________________________________________________________________
activation_18 (Activation)   (None, 254, 254, 16)      0         
_________________________________________________________________
conv2d_19 (Conv2D)           (None, 252, 252, 32)      4640      
_________________________________________________________________
batch_normalization_19 (Batc (None, 252, 252, 32)      128       
_________________________________________________________________
activation_19 (Activation)   (None, 252, 252, 32)      0         
_________________________________________________________________
conv2d_20 (Conv2D)           (None, 250, 250, 64)     

## Model Training

In [ ]:
# Function for Model training

def train_model(model,
                optimizer,
                train_data,
                test_data,
                epochs,
                callbacks=None
                ):
    """
    Function to train and validate network
    :param model: keras model object,
    :param optimizer: keras optimizer object
    :param train_data: tf dataset object
    :param test_data: tf dataset object
    :param epochs: int
    :param callbacks: list, of callbacks options
    :return: history, model
    """
    model.compile(optimizer=optimizer,loss='categorical_crossentropy', metrics=['accuracy'])

    history = model.fit_generator(train_data,
                                  epochs=epochs,
                                  validation_data=test_data,
                                  verbose=1,
                                  callbacks = callbacks
                                  )

    return history, model

In [ ]:
history_v1, model_v1 = train_model(model = model_v1,
                                   optimizer = Adam(),
                                   train_data = train_generator,
                                   test_data = test_generator,
                                   epochs = EPOCHS)

Epoch 1/50
545/545 [==============================] - 117s 216ms/step - loss: 1.1874 - accuracy: 0.4789 - val_loss: 1.3733 - val_accuracy: 0.5638
Epoch 2/50
545/545 [==============================] - 117s 215ms/step - loss: 1.0504 - accuracy: 0.5696 - val_loss: 0.9433 - val_accuracy: 0.6523
Epoch 3/50
545/545 [==============================] - 117s 215ms/step - loss: 1.0118 - accuracy: 0.5870 - val_loss: 0.8787 - val_accuracy: 0.6543
Epoch 4/50
545/545 [==============================] - 117s 215ms/step - loss: 0.9900 - accuracy: 0.5914 - val_loss: 0.8902 - val_accuracy: 0.6584
Epoch 5/50
545/545 [==============================] - 117s 215ms/step - loss: 0.9790 - accuracy: 0.5960 - val_loss: 1.7021 - val_accuracy: 0.4897
Epoch 6/50
545/545 [==============================] - 117s 215ms/step - loss: 0.9594 - accuracy: 0.6116 - val_loss: 0.8541 - val_accuracy: 0.6728
Epoch 7/50
545/545 [==============================] - 117s 215ms/step - loss: 0.9194 - accuracy: 0.6384 - val_loss: 0.7733 -